This notebook is for preparing unseen test set for evaluating dnabert and dnabert_2, by using correcponding .bed files, extracting sequences from hg19

In [1]:
# === Cell 1: imports & config ===
from __future__ import annotations
from pathlib import Path
from typing import List, Tuple, Dict, Iterable, Optional
import random
import csv
import os
import sys

# ---- Config (edit as needed) ----
POS_BED = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/K562_C_Rand_Trnc/pos_lt512.bed")
NEG_BED = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/K562/validation/K562_C_Rand_Trnc/neg_lt512.bed")
FASTA   = Path("/p/project1/hai_dnaori/piroozeh1/human_genome/hg19.fa")
# FASTA   = Path("/p/project1/hai_dnaori/piroozeh1/human_genome_ncbi/GCF_000001405.25_GRCh37.p13_genomic.fna")
OUTDIR  = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/Dataset/K562_C_Rand_Trnc/K562_C_Rand_Split/validation_test")
DEV_RATIO = 0.2
RANDOM_SEED = 42  # reproducible split
K = 6               # k-mer size

random.seed(RANDOM_SEED)


In [2]:
# === Cell 2: BED I/O ===
Interval = Tuple[str, int, int]  # (chrom, start, end) BED 0-based, half-open

def read_bed3(path: Path) -> List[Interval]:
    ivs: List[Interval] = []
    with path.open("r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if not line.strip() or line.startswith(("track", "browser", "#")):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3: 
                continue
            try:
                st = int(parts[1]); en = int(parts[2])
            except ValueError:
                continue
            if en < st: 
                continue
            ivs.append((parts[0], st, en))
    return ivs

def write_bed3(path: Path, ivs: Iterable[Interval]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as out:
        for chrom, st, en in ivs:
            out.write(f"{chrom}\t{st}\t{en}\n")

def write_bed4_labeled(path: Path, rows: Iterable[Tuple[str,int,int,int]]) -> None:
    """Write chrom,start,end,label as BED-like 4-col file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as out:
        for chrom, st, en, lab in rows:
            out.write(f"{chrom}\t{st}\t{en}\t{lab}\n")

In [3]:
# === Cell 3: merge ===


def label_rows(ivs: List[Interval], label: int) -> List[Tuple[str,int,int,int]]:
    return [(c, s, e, label) for c, s, e in ivs]

In [4]:
# === Cell 4: FASTA accessor ===
class FastaAccessor:
    """
    Random-access FASTA reader. 
    - If .fai present: true on-disk random access using offsets/line widths.
    - Else: loads all contigs into memory (RAM heavy; okay for small/reference subsets).
    """
    def __init__(self, fasta_path: Path):
        self.fasta_path = fasta_path
        self.fai_path = fasta_path.with_suffix(fasta_path.suffix + ".fai")
        self._fh = None
        self._index: Dict[str, Tuple[int, int, int, int, int]] = {}  # name -> (length, offset, line_bases, line_bytes, ???)
        self._seqs_mem: Optional[Dict[str, str]] = None
        if self.fai_path.exists():
            self._load_fai()
            self._fh = open(self.fasta_path, "rb")
        else:
            print(f"Note: {self.fai_path.name} not found; loading FASTA into memory. This may use a lot of RAM.", file=sys.stderr)
            self._load_into_memory()

    def _load_fai(self):
        with self.fai_path.open("r", encoding="utf-8") as fh:
            for line in fh:
                if not line.strip(): 
                    continue
                fields = line.rstrip("\n").split("\t")
                # name, length, offset, line_bases, line_bytes
                name = fields[0]
                length = int(fields[1]); offset = int(fields[2]); lb = int(fields[3]); lB = int(fields[4])
                self._index[name] = (length, offset, lb, lB, 0)

    def _load_into_memory(self):
        self._seqs_mem = {}
        name = None
        buf = []
        with self.fasta_path.open("r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                if line.startswith(">"):
                    if name is not None:
                        self._seqs_mem[name] = "".join(buf).replace("\n", "").replace("\r", "").upper()
                    name = line[1:].strip().split()[0]
                    buf = []
                else:
                    buf.append(line.strip())
            if name is not None:
                self._seqs_mem[name] = "".join(buf).replace("\n", "").replace("\r", "").upper()

    def fetch(self, chrom: str, start0: int, end0: int) -> str:
        """BED-like coords: 0-based start, half-open end."""
        if start0 < 0: 
            start0 = 0
        if end0 <= start0:
            return ""
        if self._seqs_mem is not None:
            seq = self._seqs_mem.get(chrom)
            if seq is None:
                return "N" * (end0 - start0)
            end0 = min(end0, len(seq))
            return seq[start0:end0].upper()
        # .fai-backed path
        if chrom not in self._index:
            return "N" * (end0 - start0)
        length, offset, lb, lB, _ = self._index[chrom]
        if start0 >= length:
            return ""
        end0 = min(end0, length)
        # Convert 0-based to 1-based inclusive for arithmetic
        start1 = start0 + 1
        end1 = end0     # since end0 is exclusive, end1 is inclusive length==end0
        # Calculate file positions considering line wraps
        # For a given 1-based position p: 
        #   line_idx = (p-1) // lb
        #   pos_in_line = (p-1) % lb
        #   file_pos = offset + line_idx * lB + pos_in_line
        def file_pos(p1: int) -> int:
            line_idx = (p1 - 1) // lb
            pos_in_line = (p1 - 1) % lb
            return offset + line_idx * lB + pos_in_line
        start_fp = file_pos(start1)
        end_fp_excl = file_pos(end1) + 1  # exclusive
        # Read chunk; remove newlines
        self._fh.seek(start_fp)
        raw = self._fh.read(end_fp_excl - start_fp)
        seq = raw.replace(b"\n", b"").replace(b"\r", b"").upper()
        # Slice to exact length in case of line-boundary issues
        want = end0 - start0
        return seq[:want].decode("ascii", errors="ignore")

    def close(self):
        if self._fh:
            self._fh.close()


In [5]:
# === Cell 5: k-mer tokenization ===
def kmerize(seq: str, k: int = 6):
    
    n = len(seq)
    kmer = [seq[i:i+k] for i in range(0, n - k + 1)]
    kmers = " ".join(kmer)
    return kmers
    # return " ".join(s[i:i+k] for i in range(0, n - k + 1)

In [6]:

# === Cell 6: pipeline ===
# 1) Read inputs
assert POS_BED.exists(), f"Missing {POS_BED}"
assert NEG_BED.exists(), f"Missing {NEG_BED}"
assert FASTA.exists(), f"Missing {FASTA}"

pos_dev = read_bed3(POS_BED)
neg_dev = read_bed3(NEG_BED)





# 3) Merge with labels and save labeled .bed (4 cols)

dev_labeled   = label_rows(pos_dev, 1)   + label_rows(neg_dev, 0)

# Shuffle within split to mix classes (optional but typical)

random.shuffle(dev_labeled)


write_bed4_labeled(OUTDIR / "dev_labeled.bed",   dev_labeled)

print(f"sizes: dev pos={len(pos_dev)} neg={len(neg_dev)}")

# 4) Extract sequences and save *_with_seq.tsv




sizes: dev pos=16031 neg=16031


In [7]:
fa = FastaAccessor(FASTA)

def write_with_seq_tsv(path: Path, rows: List[Tuple[str,int,int,int]]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter="\t")
        w.writerow(["chrom", "start", "end", "label", "seq"])
        for chrom, st, en, lab in rows:
            seq = fa.fetch(chrom, st, en)
            # pad with 'N' if fetch shorter than requested (off-end)
            if len(seq) < (en - st):
                seq = seq + ("N" * ((en - st) - len(seq)))
            w.writerow([chrom, st, en, lab, seq])




write_with_seq_tsv(OUTDIR / "dev_with_seq.tsv",   dev_labeled)

fa.close()

# 5) 6-mer tokenize and save only sequence,label as TSV
def write_kmer_tsv(in_path: Path, out_path: Path, k: int = 6):
    with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", newline="", encoding="utf-8") as fout:
        r = csv.DictReader(fin, delimiter="\t")
        w = csv.writer(fout, delimiter="\t")
        w.writerow(["sequence", "label"])
        for row in r:
            seq = row["seq"]
            lab = row["label"]
            toks = kmerize(seq, k=6)
            w.writerow([toks, lab])


write_kmer_tsv(OUTDIR / "dev_with_seq.tsv",   OUTDIR / "dev_6mer.tsv",   k=K)

print("Wrote files:")
for p in [


    OUTDIR / "dev_labeled.bed",
    OUTDIR / "dev_with_seq.tsv",
    OUTDIR / "dev.tsv",
]:
    print(" -", p.resolve())


Wrote files:
 - /p/project1/hai_dnaori/piroozeh1/human_ori/Dataset/K562_C_Rand_Trnc/K562_C_Rand_Split/validation_test/dev_labeled.bed
 - /p/project1/hai_dnaori/piroozeh1/human_ori/Dataset/K562_C_Rand_Trnc/K562_C_Rand_Split/validation_test/dev_with_seq.tsv
 - /p/project1/hai_dnaori/piroozeh1/human_ori/Dataset/K562_C_Rand_Trnc/K562_C_Rand_Split/validation_test/dev.tsv


In [11]:
# Cell 1 — setup
from pathlib import Path
import pandas as pd

PATH = Path("/p/project1/hai_dnaori/piroozeh1/human_ori/Dataset/selected_data_K562/dnabert/dev.tsv")
assert PATH.exists(), f"File not found: {PATH}"


In [12]:
# Cell 2 — read & show
df = pd.read_csv(PATH, sep="\t")
display(df.head(10))
print(f"Rows (excluding header): {len(df)}")
print(f"Columns: {list(df.columns)}")


,sequence,label
0,TAACTG AACTGT ACTGTG CTGTGT TGTGTC GTGTCC TGTC...,0
1,GCTTCC CTTCCA TTCCAC TCCACA CCACAG CACAGG ACAG...,1
2,TTCCTT TCCTTT CCTTTC CTTTCT TTTCTC TTCTCT TCTC...,0
3,AATGCT ATGCTC TGCTCA GCTCAG CTCAGA TCAGAC CAGA...,1
4,ATCATG TCATGT CATGTA ATGTAG TGTAGT GTAGTA TAGT...,0
5,CTCTGG TCTGGT CTGGTC TGGTCT GGTCTA GTCTAT TCTA...,0
6,GGGAAT GGAATT GAATTT AATTTA ATTTAG TTTAGA TTAG...,1
7,AAATAT AATATC ATATCA TATCAG ATCAGA TCAGAC CAGA...,0
8,NNNNNN NNNNNN NNNNNN NNNNNN NNNNNN NNNNNN NNNN...,0
9,TTCTTG TCTTGT CTTGTC TTGTCC TGTCCT GTCCTT TCCT...,1


Rows (excluding header): 14976
Columns: ['sequence', 'label']
